# Machine Learning project in SoSe 2025 at HTW Saar
## Idea
The goal of this project is predicting the genre(s) of a game/bundle through its given description(s)

## Dataset
For our project we use a Steam Dataset provided on moodle, since it has all information we plan on using.
The Dataset has been cut to only 2000 data points to be runnable on weaker devices.

In [ ]:
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(transform_output="pandas")

dataset = pd.read_csv("./games_march2025_cleaned_2k.csv",sep=",")
print(dataset.head())

## Preparation of the Dataset
### Removing Uniques
We would remove the following features from the Training-Set as they can/could uniquely identify a datapoint, but we don't as they will be removed in the next step anyway
- AppId
- Name of the Game
- Realease Date
- Reviews
- Header Image
- Website
- Support URL
- Support Email
- MetaCritic URL
- Developer
- Publisher
- Screenshots
- Movies
- Estimated Owners

In [ ]:
#dataset.drop(['appid', 'name', 'release_date', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'metacritic_url', 'notes', 'developers', 'publishers', 'screenshots', 'movies', 'estimated_owners'], axis=1, inplace=True)
#print(dataset.head())

## Hold onto necessary information
Our model should turn a textual description of a game into its genre. For that we need all the textual information a game has, as well as the genres of the game.
We use a ColumnTransformer to drop all unnecessary lines, merge all descriptions of a game into one big description and hold onto the genres

It is important to use ``verbose_feature_names_out=False`` so the feature names don't get changed

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

# desc, genres
column_transformer = ColumnTransformer([
        # merge all descriptions
        ('desc', FunctionTransformer(lambda X: X.fillna('').agg(' '.join, axis=1).to_frame(name="desc")),
            ['detailed_description', 'about_the_game', 'short_description']),
        ('pass', 'passthrough', ['genres']),
    ],
    verbose_feature_names_out=False
)
dataset = column_transformer.fit_transform(dataset)
print(dataset.head())

### Adding missing Information
Some Games might not have any descriptions. For these we Input an Empty String
**TODO: check if dropna and fillna numeric_only is needed, as we dont have any numbers**

In [ ]:
# missing numeric values => mean
dataset.fillna(dataset.mean(numeric_only=True), inplace=True)
# missing strings => empty string?
dataset.fillna('', inplace=True)
# drop all lines with missing values
dataset.dropna(inplace=True)

## Transform Genres
The genre information currently is a string holding a python array of genres. While this is machine-readable, we need One-Hot-Encoding for our model to work.

#### Serializing the String-Array
The "ast" library can interpret python strings as python code, and as such will be used for serializing the genres.

In [ ]:
import ast

dataset['genres'] = dataset['genres'].map(lambda s: ast.literal_eval(s))
print(dataset['genres'])

#### One-Hot-Encoding an Python-Array
The sklearn ``OneHotEncoder()`` is only able to work with an 1D Array of different classes, such as ``['Politics', 'Sport', 'Culture']``. Every datapoint can only have one concurrent classification.
Steam allows an app/bundle to have multiple genres. As such, our dataset has an 2D Array of different classes, which sklearn's ``MultiLabelBinarizer()`` does support.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_genres = MultiLabelBinarizer()
genres_encoded = mlb_genres.fit_transform(dataset.pop('genres'))
genres_df = pd.DataFrame(genres_encoded, columns=mlb_genres.classes_)
print(genres_df.head())

With this, our target matrix is completed.

### Structurizing Text
If we want our Model to be able to use text as an input, we have to vectorize the text. TF-IDF (Inverse Document Frequency) is an easy way of transforming each word into a feature with a 0 to 1 value. **TODO: filter out stopwords**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(dataset['desc']) # matrix, not pandas df
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(tfidf_df.head())

With this our feature matrix is completed

In [ ]:
X = tfidf_df
y = genres_df

## The Model

####  Removing unpredicatble Datapoints
Some Datapoints don't have a genre assigned (all feature values in y are 0). The model we use can't handle such cases, thus they have to be removed.
We filter after all values that we can use with a mask, and apply that mask to our matrices.

In [ ]:
mask = y.sum(axis=1).map(lambda x: x > 0)
print((mask == False).sum()) # count of unpredictable datapoints

X_clean = X[mask]
y_clean = y[mask]

# Splitting up data
We have to split up our data into training and testing data.
Using random_state=0 guarantees reproducability.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_clean, y_clean, random_state=0)

# Model Selection
**TODO Deciding which model to use for this task**

As a game can have multiple genres, our Model(s) has to be capable of multi-label-classification. sklearn's ``MultiOutputClassifier`` can do this. As a backend for ``MultiOutputClassifier`` we use ``LogisticRegression``

In [ ]:
# n_jobs=1 since there seems to be some multithreading join issue in sklearn (or my pc is too bad)
multi_target_clf = MultiOutputClassifier(LogisticRegression(max_iter=1337, random_state=0), n_jobs=1)

multi_target_clf.fit(X_train, y_train)

y_pred = multi_target_clf.predict(X_test)

# Evaluation
**TODO Test the Model with the test data**

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0.0))

# Optimization
**TODO optimize the model based on the test results**

# Validation
**TODO Predict actual values**

# Conclusion and outlook
**TODO Write a conclusion and outlook what can be done and where the issues were.**